# Human Codebooks Normalization, Cleaning, and Spelling Fixes

This notebook cleans all workbooks under `Codebooks/Human Codebooks` by applying conservative, auditable text normalization:

- Trim leading/trailing whitespace
- Collapse repeated internal whitespace
- Normalize smart punctuation and stray unicode spacing
- Apply conservative typo corrections from a controlled dictionary
- Preserve formulas and non-string cells
- Save in place and export a full change log

This is intentionally conservative to avoid changing coding semantics.


In [ ]:
from pathlib import Path
import re
import json
import csv
from datetime import datetime
from openpyxl import load_workbook

ROOT = Path.cwd()
if ROOT.name != 'Gates-Manfluencer-Project':
    ROOT = Path('/Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project')

HUMAN_DIR = ROOT / 'Codebooks' / 'Human Codebooks'
REPORT_DIR = ROOT / 'Codebooks' / 'Human Codebooks Cleaning Reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('HUMAN_DIR exists:', HUMAN_DIR.exists())
print('REPORT_DIR:', REPORT_DIR)

In [ ]:
# Conservative typo map: only high-confidence, low-risk fixes.
TYPO_MAP = {
    'teh': 'the',
    'recieve': 'receive',
    'seperate': 'separate',
    'occured': 'occurred',
    'occurence': 'occurrence',
    'womens': "women's",
    'mens ': "men's ",
    'behavour': 'behaviour',
    'behaviourl': 'behavioural',
    'comparision': 'comparison',
    'responce': 'response',
    'definately': 'definitely',
    'intrest': 'interest',
    'goverment': 'government',
    'famliy': 'family',
    'feminity': 'femininity',
    'masulinity': 'masculinity',
    'masculinty': 'masculinity',
    'genderd': 'gendered',
    'norms ': 'norms ',
}

SMART_REPLACEMENTS = {
    ' ': ' ',  # non-breaking space
    '​': '',   # zero-width space
    '‘': "'",
    '’': "'",
    '“': '"',
    '”': '"',
    '–': '-',
    '—': '-',
}

WS_RE = re.compile(r'\s+')


def normalize_text(s: str):
    original = s

    # smart/unicode cleanup
    for k, v in SMART_REPLACEMENTS.items():
        s = s.replace(k, v)

    # strip and collapse whitespace
    s = s.strip()
    s = WS_RE.sub(' ', s)

    # typo replacement at word boundaries (case-insensitive, preserve simple case)
    for wrong, right in TYPO_MAP.items():
        pattern = re.compile(rf'\b{re.escape(wrong.strip())}\b', re.IGNORECASE)
        def repl(m):
            w = m.group(0)
            if w.isupper():
                return right.upper()
            if w.istitle():
                return right.title()
            return right
        s = pattern.sub(repl, s)

    return s, (s != original)


In [ ]:
def clean_workbook(path: Path):
    wb = load_workbook(path)
    changes = []

    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                val = cell.value
                if isinstance(val, str):
                    # Do not touch formula-like strings
                    if val.startswith('='):
                        continue
                    new_val, changed = normalize_text(val)
                    if changed:
                        changes.append({
                            'file': str(path),
                            'sheet': ws.title,
                            'cell': cell.coordinate,
                            'before': val,
                            'after': new_val,
                        })
                        cell.value = new_val

    if changes:
        wb.save(path)

    return changes

all_files = sorted(HUMAN_DIR.rglob('*.xlsx'))
print('Workbooks found:', len(all_files))

all_changes = []
for f in all_files:
    c = clean_workbook(f)
    all_changes.extend(c)
    print(f'- {f.name}: {len(c)} changes')

print('Total changes:', len(all_changes))

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_path = REPORT_DIR / f'human_codebooks_cleaning_changes_{stamp}.json'
csv_path = REPORT_DIR / f'human_codebooks_cleaning_changes_{stamp}.csv'

json_path.write_text(json.dumps(all_changes, ensure_ascii=False, indent=2))

with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['file', 'sheet', 'cell', 'before', 'after'])
    writer.writeheader()
    writer.writerows(all_changes)

summary_path = REPORT_DIR / f'human_codebooks_cleaning_summary_{stamp}.txt'
summary_path.write_text(
    '
'.join([
        f'workbooks={len(all_files)}',
        f'total_changes={len(all_changes)}',
        f'json_report={json_path}',
        f'csv_report={csv_path}',
    ]),
    encoding='utf-8'
)

print('Reports written:')
print('-', json_path)
print('-', csv_path)
print('-', summary_path)


## Notes

- This workflow is conservative by design.
- If you want aggressive spelling correction with NLP libraries, add a second pass after manual review of this report.
